In [14]:
import pandas as pd
import numpy as np
from math import acos, degrees
from warnings import simplefilter
simplefilter(action="ignore", category=pd.errors.PerformanceWarning)

In [47]:
FILENAME = 'DJI_0057.csv'
FILE_PATH = f'csv_files/{FILENAME}'
MAP_KEYPOINTS = {0:'nose', 1:'leye', 2:'reye', 3:'lear', 4:'rear', 5:'lshoulder', 6:'rshoulder', 7:'lelbow', 8:'relbow',
                 9:'lwrist', 10:'rwrist', 11:'lhip', 12:'rhip', 13:'lknee', 14:'rknee', 15:'lankle', 16:'rankle'}
SPEED_CALC_INTERVAL = 29
MAP_DIRECTION = {-1: 'UNKNOWN', 0: 'LEFT', 1: 'RIGHT', 2: 'UP', 3: 'DOWN'}


In [48]:
df = pd.read_csv(FILE_PATH, header=None)
print(df.shape)
for i in range(3,df.shape[1]):
    df[i] =  df[i].apply(lambda x: x.replace('[','').replace(']','')) 

(846, 37)


In [49]:
# only applicable when keypoint 0 is column '3'
# keypoint 0 -> column 3 and 4
# keypoint 1 -> column 5 and 6
# keypoint 2 -> column 7 and 8

In [50]:
def distance(ax,ay,bx,by):
    return np.sqrt((ax-bx)**2+(ay-by)**2)

def angle(ax,ay,bx,by,cx,cy):
    import math
    ang = degrees(math.atan2(cy-by, cx-bx) - math.atan2(ay-by, ax-bx))
    return ang + 360 if ang < 0 else ang


In [51]:
distance_features = []
for i in range(17):
    for j in range(i+1,17):
        # x, y
        ax, ay = df[i*2+3].astype(float),df[i*2+4].astype(float)
        bx, by = df[j*2+3].astype(float),df[j*2+4].astype(float)
        colname = MAP_KEYPOINTS[i] + '_' + MAP_KEYPOINTS[j]
        df[colname] = distance(ax,ay,bx,by)
        distance_features.append(colname)

In [52]:
angle_features = []
for i in range(17):
    for j in range(i+1,17):
        for k in range(j+1, 17):
            # x, y
            ax, ay = df[i*2+3].astype(float),df[i*2+4].astype(float)
            bx, by = df[j*2+3].astype(float),df[j*2+4].astype(float)
            cx, cy = df[k*2+3].astype(float),df[k*2+4].astype(float)
            colname = MAP_KEYPOINTS[i] + '_' + MAP_KEYPOINTS[j] + '_' + MAP_KEYPOINTS[k]
            angle_array = []
            for idx in range(len(ax)):
                angle_array.append(angle(ax[idx],ay[idx],bx[idx],by[idx],cx[idx],cy[idx]))
            df[colname] = angle_array
            angle_features.append(colname)

In [53]:
df.rename(columns={0:'red_marker', 1:'direction', 2:'speed'}, inplace=True)
df = df[['red_marker','direction','speed']+distance_features+angle_features]


In [54]:
groups = df.groupby(np.arange(len(df.index))//SPEED_CALC_INTERVAL)
rows = []
for (frameno, frame) in groups:
    red_marker_ = np.any(frame['red_marker'])
    direction_ = max(frame['direction'])
    speed_ = max(frame['speed'])
    mean_ = list(frame.iloc[:,3:].mean())
    rows.append([red_marker_, direction_, speed_] + mean_)
    # print(mean_)
    # break
    # print(red_marker_)

In [55]:
final_df = pd.DataFrame(rows, columns=['red_marker','direction','speed']+distance_features+angle_features)
final_df['direction'] = final_df['direction'].map(lambda x: MAP_DIRECTION[x])

final_df['time'] = np.arange(0,final_df.shape[0]*0.5,0.5)
final_df['time'] = final_df['time'].map(lambda x: f'{x//60}:{x%60}')
final_df = final_df[['time','red_marker','direction','speed']+distance_features+angle_features]

In [56]:
for i in angle_features:
    final_df[i] = np.unwrap(final_df[i], period=360)


In [57]:
final_df.to_csv(f'cleaned_csv_files/raw_{FILENAME}')

In [58]:
final_df['speed_pct_change'] = final_df['speed'].pct_change()

distance_features_pct = [i + '_pct' for i in distance_features]
for i,j in zip(distance_features_pct,distance_features):
    final_df[i] = final_df[j].pct_change()


angle_features_pct = [i + '_degree' for i in angle_features]
for i,j in zip(angle_features_pct,angle_features):
    final_df[i] = final_df[j].diff()

In [59]:
def find_top_correlation(row, top_k=3):
    sorted_idx = np.argsort(row)
    values_max = row[sorted_idx[-top_k:]]
    colnames_max = row.index[sorted_idx[-top_k:]]
    text_max = [f'{col}:{val:.2f}' for col,val in zip(colnames_max, values_max)]
    text_max = ','.join(text_max)

    values_min = row[sorted_idx[:top_k]]
    colnames_min =  row.index[sorted_idx[:top_k]]
    text_min = [f'{col}:{val:.2f}' for col,val in zip(colnames_min, values_min)]
    text_min = ','.join(text_min)
    # return f'{colname_max}:{value_max}, {colname_min}:{value_min}'
    return text_max + text_min

In [60]:
# final_df['correlation'] = final_df[distance_features_pct+angle_features_pct].apply(lambda row: find_top_correlation(row), axis=1)
final_df['distance_correlation'] = final_df[distance_features_pct].apply(lambda row: find_top_correlation(row), axis=1)
final_df['angle_correlation'] = final_df[angle_features_pct].apply(lambda row: find_top_correlation(row), axis=1)

c:\Users\Admin\AppData\Local\Programs\Python\Python312\Lib\site-packages\numpy\core\fromnumeric.py:59: FutureWarning: The behavior of Series.argsort in the presence of NA values is deprecated. In a future version, NA values will be ordered last instead of set to -1.
  return bound(*args, **kwds)
C:\Users\Admin\AppData\Local\Temp\ipykernel_19980\1050814400.py:3: FutureWarning: Series.__getitem__ treating keys as positions is deprecated. In a future version, integer keys will always be treated as labels (consistent with DataFrame behavior). To access a value by position, use `ser.iloc[pos]`
  values_max = row[sorted_idx[-top_k:]]
C:\Users\Admin\AppData\Local\Temp\ipykernel_19980\1050814400.py:8: FutureWarning: Series.__getitem__ treating keys as positions is deprecated. In a future version, integer keys will always be treated as labels (consistent with DataFrame behavior). To access a value by position, use `ser.iloc[pos]`
  values_min = row[sorted_idx[:top_k]]
C:\Users\Admin\AppData\Loc

In [61]:
final_df['angle_correlation']

0     rknee_lankle_rankle_degree:nan,rknee_lankle_ra...
1     reye_lwrist_rknee_degree:158.21,nose_leye_lhip...
2     lear_rwrist_rknee_degree:146.49,lear_rwrist_ra...
3     nose_leye_rankle_degree:170.96,leye_lwrist_ran...
4     nose_lshoulder_relbow_degree:110.28,lear_lshou...
5     reye_rwrist_lhip_degree:122.70,nose_leye_lankl...
6     nose_rshoulder_lelbow_degree:133.18,reye_rwris...
7     reye_lwrist_rankle_degree:122.03,rear_lwrist_l...
8     nose_lwrist_rknee_degree:97.47,leye_lwrist_rkn...
9     leye_lwrist_rhip_degree:37.36,leye_lear_lelbow...
10    nose_lwrist_rknee_degree:151.43,rear_lwrist_lk...
11    leye_rear_rwrist_degree:174.17,lshoulder_lelbo...
12    rshoulder_rwrist_lhip_degree:82.14,relbow_rwri...
13    reye_lear_rear_degree:155.72,nose_reye_lknee_d...
14    reye_lear_rwrist_degree:101.28,relbow_rwrist_l...
15    lelbow_lwrist_rknee_degree:128.01,lshoulder_lw...
16    lshoulder_rshoulder_lelbow_degree:104.55,lelbo...
17    rshoulder_lelbow_relbow_degree:128.78,relb

In [62]:
final_df[['time','red_marker','direction','speed','speed_pct_change','distance_correlation','angle_correlation']].to_csv(f'cleaned_csv_files/{FILENAME}')

In [ ]:
# print(final_df.shape)
# final_df.to_csv(f'cleaned_csv_files/{FILENAME}')


(30, 820)


In [45]:
a = pd.DataFrame(data={'a':[9,6,6,1,2,39,282,2]})
a['pct_change'] = a['a'].pct_change()

In [46]:
a

,a,pct_change
0,9,NaN
1,6,-0.333333
2,6,0.000000
3,1,-0.833333
4,2,1.000000
5,39,18.500000
6,282,6.230769
7,2,-0.992908
